# 04b — Full ablation training (ShiLU-SwishTanh vs GELU vs ReLU)

Train three models with identical hyperparameters and save checkpoints/embeddings/gamma/logs.


## Install dependencies


In [23]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torch_geometric', 'scikit-learn', 'matplotlib'])


0

## Setup


In [24]:
from pathlib import Path
import os, sys, json, time, random
import numpy as np
import pandas as pd

USE_DRIVE = True
SEED = 42

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/BulkCellGNN_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, '/content/drive/MyDrive/BulkCellGNN_data')

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)

print('DATA_DIR =', DATA_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import bulkcell_gnn as bcg
import bulkcell_gnn_shilu as bcg_shilu

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR = /content/drive/MyDrive/BulkCellGNN_data
device = cuda


## Load data and graph artifacts


In [25]:
bulk_x = torch.tensor(np.load(DATA_DIR / 'bulk_expr_sym.npy'), dtype=torch.float32)
cell_x = torch.tensor(np.load(DATA_DIR / 'cell_expr_full.npy'), dtype=torch.float32)
labels = torch.tensor(np.load(DATA_DIR / 'bulk_labels.npy'), dtype=torch.long)
cell_types = torch.tensor(np.load(DATA_DIR / 'cell_types_full.npy'), dtype=torch.long)

edge_BB = torch.load(DATA_DIR / 'edge_BB_full.pt', map_location='cpu')
edge_CC = torch.load(DATA_DIR / 'edge_CC_full.pt', map_location='cpu')
edge_BC = torch.load(DATA_DIR / 'edge_BC_full.pt', map_location='cpu')
type_names = json.loads((DATA_DIR / 'cell_type_names_full.json').read_text(encoding='utf-8'))
print('bulk:', bulk_x.shape, 'cell:', cell_x.shape, 'n_types:', len(type_names))


bulk: torch.Size([536, 22880]) cell: torch.Size([47285, 5000]) n_types: 5


## Stratified split


In [26]:
idx = np.arange(bulk_x.shape[0])
tr_idx, va_idx = train_test_split(idx, test_size=0.2, stratify=labels.numpy(), random_state=SEED)
train_mask = torch.zeros(bulk_x.shape[0], dtype=torch.bool); train_mask[tr_idx] = True
val_mask = torch.zeros(bulk_x.shape[0], dtype=torch.bool); val_mask[va_idx] = True
np.save(DATA_DIR / 'train_mask_full.npy', train_mask.numpy())
np.save(DATA_DIR / 'val_mask_full.npy', val_mask.numpy())
print('train', int(train_mask.sum()), 'val', int(val_mask.sum()))


train 428 val 108


## Model constructors


In [27]:
import bulkcell_gnn_shilu as bcg_shilu
print("Loaded from:", bcg_shilu.__file__)
import inspect
print(inspect.getsource(bcg_shilu.count_shilu_params))

Loaded from: /content/drive/MyDrive/BulkCellGNN_data/bulkcell_gnn_shilu.py
def count_shilu_params(model):
    total = sum(p.numel() for p in model.parameters())
    shilu = sum(p.numel() for n, p in model.named_parameters()
                if any(n.endswith(x) for x in ('.alpha', '.gamma', '.delta')))
    print(f"Total params: {total:,}")
    print(f"ShiLU params: {shilu} ({100*shilu/total:.2f}% of total)")
    return shilu



In [28]:
def make_gelu_model():
    return bcg.BulkCellGNN(bulk_x.shape[1], cell_x.shape[1], d_latent=256, n_classes=2, n_cell_types=len(type_names), n_layers=2, dropout=0.3, cell_type_names=type_names)

def make_shilu_model():
    return bcg_shilu.BulkCellGNN(bulk_x.shape[1], cell_x.shape[1], d_latent=256, n_classes=2, n_cell_types=len(type_names), n_layers=2, dropout=0.3, cell_type_names=type_names)

def replace_gelu_with_relu(module):
    for name, child in module.named_children():
        if isinstance(child, nn.GELU):
            setattr(module, name, nn.ReLU())
        else:
            replace_gelu_with_relu(child)

def make_relu_model():
    m = make_gelu_model()
    replace_gelu_with_relu(m)
    return m


## Train utility (shared across variants)


In [29]:
def train_variant(model_name, model, save_ckpt_name, save_hb_name, save_hc_name, save_gamma_name, track_shilu=False):
    t_start = time.time()
    model = model.to(device)
    criterion = bcg.BulkCellLoss(lambda_cls=1.0, lambda_rec=0.5, lambda_aln=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

    bulk_x_d, cell_x_d = bulk_x.to(device), cell_x.to(device)
    labels_d = labels.to(device)
    cell_types_d = cell_types.to(device)
    edge_BB_d, edge_CC_d, edge_BC_d = edge_BB.to(device), edge_CC.to(device), edge_BC.to(device)
    train_mask_d, val_mask_d = train_mask.to(device), val_mask.to(device)

    logs = {
        'epoch': [], 'loss_total': [], 'loss_cls': [], 'loss_recon': [], 'loss_align': [],
        'val_auc': [], 'best_auc': 0.0, 'best_epoch': 0, 'training_time_sec': 0.0
    }
    if track_shilu:
        logs['shilu_params'] = {}

    for epoch in range(1, 101):
        model.train()
        optimizer.zero_grad()
        out = model(bulk_x_d, cell_x_d, edge_BB_d, edge_CC_d, edge_BC_d, cell_types_d, return_gamma=True)
        L_total, loss_dict = criterion(out['logits'][train_mask_d], labels_d[train_mask_d], out['cell_recon'], cell_x_d, out['h_B'], out['h_C'], edge_BC_d)
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            outv = model(bulk_x_d, cell_x_d, edge_BB_d, edge_CC_d, edge_BC_d, cell_types_d, return_gamma=True)
            probs = F.softmax(outv['logits'][val_mask_d], dim=-1)[:, 1].detach().cpu().numpy()
            y = labels_d[val_mask_d].detach().cpu().numpy()
            auc = float(roc_auc_score(y, probs))

        logs['epoch'].append(epoch)
        logs['loss_total'].append(float(loss_dict['total']))
        logs['loss_cls'].append(float(loss_dict['classify']))
        logs['loss_recon'].append(float(loss_dict['recon']))
        logs['loss_align'].append(float(loss_dict['align']))
        logs['val_auc'].append(auc)

        if auc > logs['best_auc']:
            logs['best_auc'] = auc
            logs['best_epoch'] = epoch
            torch.save(model.state_dict(), DATA_DIR / save_ckpt_name)

        if track_shilu:
            for n, p in model.named_parameters():
                if any(n.endswith(k) for k in ('.alpha', '.gamma', '.delta')):
                    logs['shilu_params'].setdefault(n, []).append(float(p.detach().cpu().item()))


        print(f'[{model_name}] epoch {epoch:03d} loss={loss_dict["total"]:.4f} cls={loss_dict["classify"]:.4f} val_auc={auc:.4f}')

    model.load_state_dict(torch.load(DATA_DIR / save_ckpt_name, map_location=device))
    model.eval()
    with torch.no_grad():
        out_final = model(bulk_x_d, cell_x_d, edge_BB_d, edge_CC_d, edge_BC_d, cell_types_d, return_gamma=True)

    torch.save(out_final['h_B'].detach().cpu(), DATA_DIR / save_hb_name)
    torch.save(out_final['h_C'].detach().cpu(), DATA_DIR / save_hc_name)
    np.save(DATA_DIR / save_gamma_name, out_final['gamma'].detach().cpu().numpy())

    logs['training_time_sec'] = float(time.time() - t_start)
    return logs


## Train ShiLU model


In [30]:
shilu_model = make_shilu_model()
try:
    bcg_shilu.count_shilu_params(shilu_model)
except Exception as e:
    print('count_shilu_params unavailable:', e)
logs_shilu = train_variant('ShiLU-SwishTanh', shilu_model, 'model_best_shilu.pt', 'hB_shilu.pt', 'hC_shilu.pt', 'gamma_shilu.npy', track_shilu=True)


Total params: 20,696,358
ShiLU params: 18 (0.00% of total)
[ShiLU-SwishTanh] epoch 001 loss=1.0436 cls=0.6441 val_auc=0.9735
[ShiLU-SwishTanh] epoch 002 loss=0.7642 cls=0.3338 val_auc=0.9789
[ShiLU-SwishTanh] epoch 003 loss=0.6609 cls=0.2429 val_auc=0.9857
[ShiLU-SwishTanh] epoch 004 loss=0.4913 cls=0.1074 val_auc=0.9864
[ShiLU-SwishTanh] epoch 005 loss=0.4367 cls=0.1052 val_auc=0.9878
[ShiLU-SwishTanh] epoch 006 loss=0.3560 cls=0.0841 val_auc=0.9932
[ShiLU-SwishTanh] epoch 007 loss=0.2749 cls=0.0578 val_auc=0.9946
[ShiLU-SwishTanh] epoch 008 loss=0.2252 cls=0.0502 val_auc=0.9959
[ShiLU-SwishTanh] epoch 009 loss=0.1948 cls=0.0485 val_auc=0.9980
[ShiLU-SwishTanh] epoch 010 loss=0.1636 cls=0.0358 val_auc=0.9980
[ShiLU-SwishTanh] epoch 011 loss=0.1468 cls=0.0307 val_auc=0.9973
[ShiLU-SwishTanh] epoch 012 loss=0.1320 cls=0.0238 val_auc=0.9973
[ShiLU-SwishTanh] epoch 013 loss=0.1222 cls=0.0193 val_auc=0.9959
[ShiLU-SwishTanh] epoch 014 loss=0.1159 cls=0.0170 val_auc=0.9932
[ShiLU-SwishTanh]

## Train GELU model


In [31]:
gelu_model = make_gelu_model()
logs_gelu = train_variant('GELU', gelu_model, 'model_best_gelu.pt', 'hB_gelu.pt', 'hC_gelu.pt', 'gamma_gelu.npy')


[GELU] epoch 001 loss=1.0179 cls=0.6233 val_auc=0.9728
[GELU] epoch 002 loss=0.7000 cls=0.2840 val_auc=0.9796
[GELU] epoch 003 loss=0.5407 cls=0.1428 val_auc=0.9871
[GELU] epoch 004 loss=0.4556 cls=0.0933 val_auc=0.9939
[GELU] epoch 005 loss=0.3731 cls=0.0652 val_auc=0.9939
[GELU] epoch 006 loss=0.2987 cls=0.0530 val_auc=0.9952
[GELU] epoch 007 loss=0.2298 cls=0.0455 val_auc=0.9952
[GELU] epoch 008 loss=0.1701 cls=0.0322 val_auc=0.9946
[GELU] epoch 009 loss=0.1384 cls=0.0264 val_auc=0.9939
[GELU] epoch 010 loss=0.1248 cls=0.0259 val_auc=0.9939
[GELU] epoch 011 loss=0.1137 cls=0.0202 val_auc=0.9932
[GELU] epoch 012 loss=0.1120 cls=0.0220 val_auc=0.9932
[GELU] epoch 013 loss=0.1032 cls=0.0149 val_auc=0.9932
[GELU] epoch 014 loss=0.1029 cls=0.0153 val_auc=0.9918
[GELU] epoch 015 loss=0.0981 cls=0.0103 val_auc=0.9925
[GELU] epoch 016 loss=0.0960 cls=0.0074 val_auc=0.9918
[GELU] epoch 017 loss=0.0945 cls=0.0068 val_auc=0.9912
[GELU] epoch 018 loss=0.0889 cls=0.0045 val_auc=0.9898
[GELU] epo

## Train ReLU model


In [32]:
relu_model = make_relu_model()
logs_relu = train_variant('ReLU', relu_model, 'model_best_relu.pt', 'hB_relu.pt', 'hC_relu.pt', 'gamma_relu.npy')


[ReLU] epoch 001 loss=1.0883 cls=0.6788 val_auc=0.9769
[ReLU] epoch 002 loss=0.7027 cls=0.2746 val_auc=0.9885
[ReLU] epoch 003 loss=0.5101 cls=0.1317 val_auc=0.9946
[ReLU] epoch 004 loss=0.4040 cls=0.0891 val_auc=0.9959
[ReLU] epoch 005 loss=0.3200 cls=0.0683 val_auc=0.9959
[ReLU] epoch 006 loss=0.2405 cls=0.0502 val_auc=0.9959
[ReLU] epoch 007 loss=0.1835 cls=0.0423 val_auc=0.9952
[ReLU] epoch 008 loss=0.1511 cls=0.0358 val_auc=0.9952
[ReLU] epoch 009 loss=0.1367 cls=0.0312 val_auc=0.9946
[ReLU] epoch 010 loss=0.1283 cls=0.0258 val_auc=0.9973
[ReLU] epoch 011 loss=0.1228 cls=0.0218 val_auc=0.9973
[ReLU] epoch 012 loss=0.1222 cls=0.0203 val_auc=0.9952
[ReLU] epoch 013 loss=0.1163 cls=0.0111 val_auc=0.9946
[ReLU] epoch 014 loss=0.1172 cls=0.0118 val_auc=0.9939
[ReLU] epoch 015 loss=0.1093 cls=0.0082 val_auc=0.9939
[ReLU] epoch 016 loss=0.0995 cls=0.0059 val_auc=0.9946
[ReLU] epoch 017 loss=0.0897 cls=0.0054 val_auc=0.9939
[ReLU] epoch 018 loss=0.0839 cls=0.0070 val_auc=0.9939
[ReLU] epo

## Save training logs


In [33]:
all_logs = {'shilu': logs_shilu, 'gelu': logs_gelu, 'relu': logs_relu}
(DATA_DIR / 'training_logs_ablation.json').write_text(json.dumps(all_logs, indent=2), encoding='utf-8')
print('Saved training_logs_ablation.json')


Saved training_logs_ablation.json


## Quick AUC overlay figure


In [34]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(logs_shilu['epoch'], logs_shilu['val_auc'], label='ShiLU-SwishTanh')
ax.plot(logs_gelu['epoch'], logs_gelu['val_auc'], label='GELU')
ax.plot(logs_relu['epoch'], logs_relu['val_auc'], label='ReLU')
ax.axhline(0.964, linestyle='--', color='gray', label='SVM baseline 0.964')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val AUC'); ax.set_title('Val AUC comparison')
ax.legend()
fig.savefig(DATA_DIR / 'ablation_val_auc_overlay.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)


## Timing summary table


In [35]:
summary_rows = [
    {'Model': 'ShiLU-SwishTanh', 'Training time': logs_shilu['training_time_sec'], 'Best epoch': logs_shilu['best_epoch'], 'Val AUC': logs_shilu['best_auc']},
    {'Model': 'GELU', 'Training time': logs_gelu['training_time_sec'], 'Best epoch': logs_gelu['best_epoch'], 'Val AUC': logs_gelu['best_auc']},
    {'Model': 'ReLU', 'Training time': logs_relu['training_time_sec'], 'Best epoch': logs_relu['best_epoch'], 'Val AUC': logs_relu['best_auc']},
]
summary_df = pd.DataFrame(summary_rows)
summary_df['Training time'] = summary_df['Training time'].map(lambda x: f'{x/60:.2f} min')
print(summary_df.to_markdown(index=False))
summary_df.to_csv(DATA_DIR / 'model_timing_summary.csv', index=False)


| Model           | Training time   |   Best epoch |   Val AUC |
|:----------------|:----------------|-------------:|----------:|
| ShiLU-SwishTanh | 0.50 min        |           21 |  0.998641 |
| GELU            | 0.45 min        |            6 |  0.995245 |
| ReLU            | 0.45 min        |           10 |  0.997283 |


## Final verification


In [36]:
expected = [
 'model_best_shilu.pt','model_best_gelu.pt','model_best_relu.pt',
 'hB_shilu.pt','hC_shilu.pt','hB_gelu.pt','hC_gelu.pt','hB_relu.pt','hC_relu.pt',
 'gamma_shilu.npy','gamma_gelu.npy','gamma_relu.npy',
 'training_logs_ablation.json','model_timing_summary.csv'
]
missing = []
for name in expected:
    p = DATA_DIR / name
    if p.exists():
        print('OK ', name)
    else:
        print('MISSING', name)
        missing.append(name)
if missing:
    raise FileNotFoundError('Missing outputs: ' + ', '.join(missing))
print('Notebook 04b complete')


OK  model_best_shilu.pt
OK  model_best_gelu.pt
OK  model_best_relu.pt
OK  hB_shilu.pt
OK  hC_shilu.pt
OK  hB_gelu.pt
OK  hC_gelu.pt
OK  hB_relu.pt
OK  hC_relu.pt
OK  gamma_shilu.npy
OK  gamma_gelu.npy
OK  gamma_relu.npy
OK  training_logs_ablation.json
OK  model_timing_summary.csv
Notebook 04b complete
